In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# Reemplaza 'ruta/al/archivo.xlsx' con la ruta real de tu archivo en Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/EMISION_211024_1.xlsx'

# Leer el archivo Excel
df = pd.read_excel(file_path)

# Mostrar las primeras filas del dataframe para verificar que se leyó correctamente
display(df.head())


,NRO LOTE,NRO LOTES ANTERIORES,FECHA CARGA LOTE,CÓDIGO PRODUCTO,PRODUCTO,PLAN,NOMBRE DE PLAN,CÓD. CERTIFICADO CANAL,TIPO MOVIMIENTO,NOMBRE,...,NUMERO DE POLIZA AE,FEC. INICIO AE,FEC. FIN AE,PRIMA BRUTA AE,MONTO COMISIÓN CANAL,FEC. INICIO AX,FEC. FIN AX,PRIMA BRUTA AX,TIPO DOC EMISIÓN ORIGEN,N. DOC EMISIÓN ORIGEN
0,1238602,NaN,2024-10-16 11:37:43,3588,Vida Contivida BBVA,68772,Contivida Telemarketin,C00110465204001178509,Renovacion,GAMBERTY CLEMENTE,...,B-20135,2024-08-15,2025-08-15,5.49,0,NaN,NaN,NaN,NaN,NaN
1,1238602,NaN,2024-10-16 11:37:43,3588,Vida Contivida BBVA,68772,Contivida Telemarketin,C00110465204001270065,Renovacion,LUIS HEBERT,...,B-20552,2024-01-15,2025-01-15,5.49,0,NaN,NaN,NaN,NaN,NaN
2,1238602,NaN,2024-10-16 11:37:43,3588,Vida Contivida BBVA,68772,Contivida Telemarketin,C00110465214001178622,Renovacion,JANET VICTORIA,...,B-19918,2024-08-15,2025-08-15,5.49,0,NaN,NaN,NaN,NaN,NaN
3,1238602,NaN,2024-10-16 11:37:43,3588,Vida Contivida BBVA,68772,Contivida Telemarketin,C00110465224001198399,Renovacion,ROSA ISABEL,...,B-20236,2024-01-15,2025-01-15,5.49,0,NaN,NaN,NaN,NaN,NaN
4,1238602,NaN,2024-10-16 11:37:43,3588,Vida Contivida BBVA,68772,Contivida Telemarketin,C00110465264001401704,Renovacion,LUCIO,...,B-20722,2024-08-08,2025-08-08,5.49,0,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Asegurarse de que los nombres de columna estén en minúsculas para evitar problemas de case-sensitivity
df.columns = df.columns.str.lower()

# df.loc[df['código producto'].isin([6377, 5520]), 'prima bruta'] *= 1.2154
# Agrupar y sumar 'PRIMA BRUTA AE' por 'NOMBRE ARCHIVO', 'NRO LOTE', y 'MONEDA'
suma_prima = df.groupby(['nombre archivo', 'nro lote', 'moneda'])['prima bruta'].sum().unstack(fill_value=0).reset_index()

# Agrupar y contar registros por 'NOMBRE ARCHIVO', 'NRO LOTE', y 'MONEDA'
conteo_registros = df.groupby(['nombre archivo', 'nro lote', 'moneda']).size().unstack(fill_value=0).reset_index()

# Renombrar las columnas adecuadamente
suma_prima.columns.name = None
conteo_registros.columns.name = None
suma_prima = suma_prima.rename(columns={'SOL': 'PRIMA BRUTA SOL', 'USD': 'PRIMA BRUTA USD'})
conteo_registros = conteo_registros.rename(columns={'SOL': 'Registros SOL', 'USD': 'Registros USD'})

# Combinar las dos tablas en una
final_result = pd.merge(suma_prima, conteo_registros, on=['nombre archivo', 'nro lote'])

# Asegurarse de que todos los valores NaN sean reemplazados por 0
final_result = final_result.fillna(0)

# Convertir todos los registros de 'nombre archivo' a minúsculas
final_result['nombre archivo'] = final_result['nombre archivo'].str.lower()

# Mostrar el resultado
display(final_result)

,nombre archivo,nro lote,PRIMA BRUTA SOL,PRIMA BRUTA USD,Registros SOL,Registros USD
0,20100130204_0206001_20240927_004.txt,1230797,0.000000,1280.41,0,109
1,20100130204_0206001_20240930_004.txt,1237164,0.000000,4994.19,0,369
2,20100130204_0206001_20241001_004.txt,1237163,0.000000,852.20,0,44
3,20100130204_0206001_20241002_004.txt,1237170,0.000000,1174.14,0,45
4,20100130204_0206001_20241003_004.txt,1237177,0.000000,610.17,0,47
...,...,...,...,...,...,...
118,20100130204_4121001_20241011_004.txt,1239492,63910.204672,0.00,122,0
119,20100130204_4121001_20241014_004.txt,1239511,196939.599644,0.00,374,0
120,20100130204_4121001_20241015_004.txt,1239521,62333.928104,0.00,137,0
121,20100130204_4121001_20241016_004.txt,1239548,72827.108312,0.00,132,0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#!pip install jira
from datetime import datetime
#from jira import JIRA, JIRAError
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
import json
import os
import requests

In [4]:

# --- Configuración (evita hardcodear credenciales) ---
SERVER = "https://rimacseguros.atlassian.net"
USUARIO = os.getenv("JIRA_EMAIL", "paul.sanchez@rimac.com.pe")
TOKEN = os.getenv("JIRA_API_TOKEN", "ATATT3xFfGF0CjokoacwWkFHhMkRb3hOK1mPm_1Obg4sg9GnJSjOeJgxEv54sjvmlcOFlIy9KQ5yvmLFO1b3IK8e9cGEj46QFzb3xwsO47mFyZGMm_jIiexDYOmTava_LgSKUtHUy5hKxkweG4YmHqx91mw3GehtIrbYfnpPV2RLxQFCtozFFqY=342FD831")

# --- Autenticación básica ---
session = requests.Session()
auth = (USUARIO, TOKEN)
headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}
session.auth = auth
session.headers.update(headers)

In [5]:
url_busqueda = f"{SERVER}/rest/api/3/search/jql"

JQL = 'project = PMR AND "fecha recepción[date]" >= "2024-09-01" AND "fecha recepción[date]" <= "2025-12-30"'

payload_base = {
    "jql": JQL,
    "maxResults": 100,  # tamaño de página
    "fields": ["key", "summary", "status", "assignee", "created"],  # pide lo que necesitas
    # "expand": ["renderedFields"],  # opcional
}

all_issues = []
next_page_token = None

while True:
    payload = dict(payload_base)
    if next_page_token:
        payload["nextPageToken"] = next_page_token

    resp = session.post(url_busqueda, json=payload)
    resp.raise_for_status()
    data = resp.json()

    batch = data.get("issues", [])
    all_issues.extend(batch)
    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        break

print(f"Total issues obtenidos: {len(all_issues)}")

# Mostrar los primeros issues
issue_data = [{"key": issue["key"], "summary": issue["fields"]["summary"]} for issue in all_issues]
print("Issues en el proyecto:")
for issue in issue_data[:10]:
    print(f"Key: {issue['key']}, Summary: {issue['summary']}")


Total issues obtenidos: 10523
Issues en el proyecto:
Key: PMR-15929, Summary: RED151225_1612_803.txt
Key: PMR-15928, Summary: RED121225_1512_803.txt
Key: PMR-15927, Summary: RED111225_1212_803.txt
Key: PMR-15926, Summary: RED101225_1112_803.txt
Key: PMR-15925, Summary: RED051225_1012_803.txt
Key: PMR-15924, Summary: RED041225_0512_803.txt
Key: PMR-15923, Summary: RED031225_0412_803.txt
Key: PMR-15922, Summary: RED021225_0312_803.txt
Key: PMR-15921, Summary: RED011225_0212_803.txt
Key: PMR-15920, Summary: RED281125_0112_803.txt


In [ ]:
#####################################
######ACTUALIZAR  ISSUES#############
#####################################

# Crear un diccionario para buscar issues por summary
issue_dict = {issue["fields"]["summary"]: issue["key"] for issue in all_issues}
num_issues = 0
# Actualizar issues en Jira
for _, row in final_result.iterrows():
    nombre_archivo = row['nombre archivo']
    if nombre_archivo in issue_dict:
        print(f"Actualizando issue {issue_dict[nombre_archivo]}")
        num_issues += 1
        issue_key = issue_dict[nombre_archivo]
        update_url = f"{SERVER}/rest/api/3/issue/{issue_key}"

        # Crear el payload de actualización
        update_payload = {
            "fields": {
                "customfield_10752": row["Registros SOL"],#'Total Registros Emitidos PEN'
                "customfield_10755": row["PRIMA BRUTA SOL"],#'Total Prima Bruta Emitida PEN'
                "customfield_10507": row["Registros USD"],#'Total Registros Emitidos USD'
                "customfield_10536": row["PRIMA BRUTA USD"] #'Total Prima Bruta Emitida USD'
            }
        }

        # Realizar la solicitud de actualización
        response = requests.put(update_url, headers=headers, json=update_payload, auth=auth)
        if response.status_code == 204:
            print(f"Issue {issue_key} actualizado correctamente.")
        else:
            print(f"Error al actualizar el issue {issue_key}: {response.status_code}")
            print(response.text)
print(f"Issues actualizados: {num_issues}")
print("Actualización completa.")



Actualizando issue PMR-5514
Issue PMR-5514 actualizado correctamente.
Actualizando issue PMR-5623
Issue PMR-5623 actualizado correctamente.
Actualizando issue PMR-5661
Issue PMR-5661 actualizado correctamente.
Actualizando issue PMR-5683
Issue PMR-5683 actualizado correctamente.
Actualizando issue PMR-5696
Issue PMR-5696 actualizado correctamente.
Actualizando issue PMR-5724
Issue PMR-5724 actualizado correctamente.
Actualizando issue PMR-5769
Issue PMR-5769 actualizado correctamente.
Actualizando issue PMR-5809
Issue PMR-5809 actualizado correctamente.
Actualizando issue PMR-6091
Issue PMR-6091 actualizado correctamente.
Actualizando issue PMR-6034
Issue PMR-6034 actualizado correctamente.
Actualizando issue PMR-6069
Issue PMR-6069 actualizado correctamente.
Actualizando issue PMR-5630
Issue PMR-5630 actualizado correctamente.
Actualizando issue PMR-5663
Issue PMR-5663 actualizado correctamente.
Actualizando issue PMR-5688
Issue PMR-5688 actualizado correctamente.
Actualizando issue P